# 에이전트 (Agent)

에이전트는 언어 모델(LLM)과 도구(Tool)를 결합하여 복잡한 작업을 수행하는 시스템입니다. 에이전트는 주어진 작업에 대해 추론하고, 필요한 도구를 선택하며, 목표를 향해 반복적으로 작업을 수행합니다.

![](../../notebooks/assets/langgraph-agent.png)

LangChain의 `create_agent` 함수는 프로덕션 수준의 에이전트 구현을 제공합니다. 이 함수를 사용하면 모델 선택, 도구 연동, 미들웨어 설정 등을 손쉽게 구성할 수 있습니다.

> 참고 문서: [LangChain Agents](https://docs.langchain.com/oss/python/langchain/agents)

## 환경 설정

에이전트 튜토리얼을 시작하기 전에 필요한 환경을 설정합니다. `dotenv`를 사용하여 API 키를 로드하고, `langchain_teddynote`의 로깅 기능을 활성화하여 LangSmith에서 실행 추적을 확인할 수 있도록 합니다.

LangSmith 추적을 활성화하면 에이전트의 추론 과정, 도구 호출, 응답 생성 등을 시각적으로 디버깅할 수 있어 개발에 큰 도움이 됩니다.

아래 코드는 환경 변수를 로드하고 LangSmith 프로젝트를 설정합니다.

In [1]:
from dotenv import load_dotenv
from langchain_teddynote import logging

# 환경 변수 로드
load_dotenv(override=True)
# 추적을 위한 프로젝트 이름 설정
logging.langsmith("LangGraph-V1-Tutorial")

LangSmith 추적을 시작합니다.
[프로젝트명]
LangGraph-V1-Tutorial


## 모델 (Model)

에이전트의 추론 엔진인 LLM은 `create_agent` 함수의 첫 번째 인자로 전달합니다. 모델을 지정하는 방법은 크게 두 가지가 있습니다.

### 방법 1: 문자열 식별자 (provider:model)

가장 간단한 방법은 `provider:model` 형식의 문자열을 직접 `create_agent`에 전달하는 것입니다. 이 방식은 빠른 프로토타이핑에 적합합니다.

### 방법 2: init_chat_model 함수

`init_chat_model` 함수를 사용하면 모델 인스턴스를 먼저 생성한 후 전달할 수 있습니다. 이 방식은 모델 옵션을 세밀하게 제어할 때 유용합니다.

아래 코드는 두 가지 방법을 모두 보여주는 예시입니다.

In [2]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

# 방법 1: 문자열 식별자를 직접 전달 (빠른 프로토타이핑에 적합)
# agent = create_agent("openai:gpt-5.4", tools=[])
# 방법 2: init_chat_model로 모델 인스턴스 생성 후 전달
# 다른 모델을 사용하려면 gpt-5.4-mini, claude-sonnet-5 등으로 변경하세요
model = init_chat_model("gpt-5.4")
# 모델 인스턴스를 사용하여 에이전트 생성
agent = create_agent(model, tools=[])

### 모델 세부 설정

더 세밀한 제어가 필요한 경우, 모델 생성 시 추가 옵션을 전달하여 다양한 설정을 적용할 수 있습니다.

- `temperature`: 응답의 무작위성을 제어합니다 (0에 가까울수록 일관된 응답, 1에 가까울수록 다양한 응답)
- `max_tokens`: 생성할 최대 토큰 수를 제한합니다
- `timeout`: 요청 타임아웃(초)을 설정합니다

모델 세부 설정을 적용하는 방법은 두 가지가 있습니다.

**방법 1: init_chat_model 사용**

`init_chat_model` 함수에 추가 옵션을 전달하는 방식입니다. Provider에 관계없이 동일한 인터페이스로 사용할 수 있어 편리합니다.

**방법 2: Provider별 클래스 직접 사용**(위에서 문자열 식별자 넣는 거랑 다른 얘기임!! 헷갈리실까봐 ㅋㅎ)

`ChatOpenAI`, `ChatAnthropic` 등 Provider별 클래스를 직접 인스턴스화하는 방식입니다. Provider 고유의 옵션을 세밀하게 제어할 때 유용합니다.

아래 코드는 두 가지 방법을 모두 보여주는 예시입니다.

수인 Comment: init_chat_model이 더 범용적이기는 하겠죠!

In [3]:
# 방법 1: init_chat_model에 추가 옵션 전달
# Provider에 관계없이 동일한 인터페이스로 사용 가능
model = init_chat_model(
    "gpt-5.4",  # 다른 모델을 사용하려면 gpt-5.4-mini, claude-sonnet-5 등으로 변경하세요
    temperature=0.1,  # 응답의 무작위성 제어
    max_tokens=1000,  # 최대 생성 토큰 수
    timeout=30,  # 요청 타임아웃(초)
)

agent = create_agent(model, tools=[])

In [4]:
# 방법 2: Provider별 클래스 직접 사용
# Provider 고유의 옵션을 세밀하게 제어할 때 유용
from langchain_openai import ChatOpenAI
# from langchain_anthropic import ChatAnthropic  # Anthropic 사용 시

model = ChatOpenAI(
    model="gpt-5.4",
    temperature=0.1,
    max_tokens=1000,
    timeout=30,
)

agent = create_agent(model, tools=[])

### 동적 모델 선택

동적 모델 선택은 런타임에 현재 상태와 컨텍스트를 기반으로 사용할 모델을 결정하는 패턴입니다. 이를 통해 정교한 라우팅 로직과 비용 최적화가 가능합니다. 예를 들어, 간단한 질문에는 경량 모델을, 복잡한 대화에는 고급 모델을 사용할 수 있습니다.

`wrap_model_call` 데코레이터를 사용하면 모델 호출 전에 요청을 검사하고 수정할 수 있는 미들웨어를 생성할 수 있습니다.

![](../../notebooks/assets/wrap_model_call.png)

아래 코드는 대화 길이에 따라 모델을 동적으로 선택하는 예시입니다.

### ModelRequest 속성

`ModelRequest`는 에이전트의 모델 호출 정보를 담는 데이터 클래스로, 미들웨어에서 요청을 검사하고 수정할 때 사용됩니다. `override()` 메서드를 통해 여러 속성을 동시에 변경할 수 있습니다.

아래 코드는 ModelRequest를 사용하여 동적으로 모델과 시스템 프롬프트를 변경하는 예시입니다.

> **\*참고하십셔!** 미들웨어 함수는 어떻게 생겨야 할까?
>
> `create_agent(..., middleware=[...])`의 리스트에는 일반 함수가 아니라 **미들웨어 객체**가 들어갑니다. `@wrap_model_call` 같은 데코레이터가 우리가 작성한 함수를 미들웨어 객체로 감싸 주는 것입니다. 그래서 **함수가 받는 인자와 돌려줄 값은 어떤 데코레이터를 쓰느냐에 따라 달라집니다.**
>
> | 데코레이터 | 함수가 받는 인자 | 돌려줄 값 |
> |:---|:---|:---|
> | `@wrap_model_call` | `(request, handler)` | `handler(request)`의 결과 |
> | `@wrap_tool_call` | `(request, handler)` | `handler(request)`의 결과 |
> | `@before_model` / `@after_model` | `(state, runtime)` | 바꿀 상태 `dict` 또는 `None` |
> | `@dynamic_prompt` | `(request)` | 시스템 프롬프트 문자열 |
>
> - **인자 이름보다 순서가 중요합니다.** 데코레이터는 `func(request, handler)`처럼 위치 순서대로 값을 넘깁니다. `request`, `handler`는 공식 문서에서 쓰는 관례적인 이름입니다.
>   - `request`: 모델에 보낼 요청 (`request.state`, `request.model`, `request.messages` 등)
>   - `handler`: 다음 단계(실제 모델 호출)를 실행하는 함수
> - **`-> ModelResponse`는 타입 힌트입니다.** Python은 실행할 때 타입 힌트를 검사하지 않으므로 없어도 동작합니다. 다만 붙여 두면 VS Code 자동완성이 잘 되고, 읽는 사람이 반환값을 바로 알 수 있습니다. (1주차의 `TypedDict`, `Annotated`와 같은 맥락)
> - **진짜 주의할 점은 `return handler(request)`입니다.** `handler(request)`를 호출하지 않으면 모델이 아예 호출되지 않고, 호출만 하고 `return`하지 않으면 응답이 에이전트로 돌아가지 않습니다. `handler` 호출 **전**에는 요청을 수정하고(예: 모델 교체), 호출 **후**에는 응답을 가공·기록할 수 있습니다.



In [5]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse

# 기본 모델과 고급 모델 정의
basic_model = init_chat_model("gpt-5.4-mini")
advanced_model = init_chat_model("gpt-5.4")


@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    """대화 복잡도에 따라 모델 선택"""
    message_count = len(request.state["messages"])

    # 긴 대화에는 고급 모델 사용
    if message_count > 10:
        model = advanced_model
    else:
        model = basic_model

    request.model = model
    print(f"모델 선택: {request.model.model}")
    return handler(request)


agent = create_agent(
    model=basic_model, tools=[], middleware=[dynamic_model_selection]  # 기본 모델
)

> **\*위 코드를 조금 더 자세히 이해해봅시당** `request`와 `handler`에는 뭐가 들어올까?
>
> 이 함수는 우리가 직접 호출하지 않습니다. `agent.invoke()`를 실행하면 **에이전트가 LLM을 부르기 직전에 대신 호출하면서** 두 값을 채워 넣습니다.
>
> - **`request`** = LLM에게 보낼 **주문서** (`ModelRequest` 객체)
>
>   | 속성 | 내용 |
>   |:---|:---|
>   | `model` | 이번에 쓸 LLM (여기서는 처음에 `basic_model`) |
>   | `messages` | 지금까지의 대화 기록 |
>   | `system_message` | 시스템 프롬프트 |
>   | `tools` | 모델이 쓸 수 있는 도구 목록 |
>   | `state` | 에이전트의 현재 상태 (`{"messages": [...]}`) |
>   | `runtime` | 실행 시 전달된 context 등 (2주차 `context_schema`) |
>
> - **`handler`** = 주문서를 받아 **실제로 LLM을 호출하고 답을 돌려주는 함수**입니다. 값이 아니라 함수이므로 `handler(request)`처럼 호출합니다.
>
> ```
> 에이전트 → dynamic_model_selection(request, handler) 호출
>   주문서 읽기: 대화가 10개 넘나?  →  request.model 을 고쳐 씀
>   handler(request)  →  고친 주문서대로 LLM 호출 → 답변 반환
> ```
>
> 즉, 미들웨어는 **주문서가 LLM에게 가기 직전에 가로채 고쳐 쓰는 단계**이고, `handler`는 "이제 진짜 보내기" 버튼입니다.

In [6]:
from langchain_teddynote.messages import stream_graph
from langchain.messages import HumanMessage

stream_graph(
    agent,
    inputs={
        "messages": [HumanMessage(content="머신러닝의 동작 원리에 대해서 설명해줘")]
    },
)

C:\Users\User\AppData\Local\Temp\ipykernel_15952\1540556384.py:19: DeprecationWarning: Direct attribute assignment to ModelRequest.model is deprecated. Use request.override(model=...) instead to create a new request with the modified attribute.
  request.model = model


모델 선택: gpt-5.4-mini

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
물론입니다.  
머신러닝(Machine Learning, 기계학습)은 **컴퓨터가 명시적으로 규칙을 다 입력받지 않아도, 데이터를 통해 패턴을 학습해서 예측이나 판단을 하는 방법**입니다.

## 1. 기본 아이디어
전통적인 프로그래밍은 보통 이렇게 합니다.

- **입력 데이터** + **사람이 만든 규칙** → **결과**

예:
- “이메일 제목에 ‘무료’, ‘당첨’이 있으면 스팸으로 분류”

하지만 머신러닝은 다릅니다.

- **입력 데이터** + **정답 데이터(라벨)** → **모델이 규칙을 스스로 학습**
- 그 다음
- **새로운 입력 데이터** → **학습한 모델이 예측**

즉, 사람이 규칙을 직접 다 쓰는 대신, **데이터에서 규칙을 찾아내게 하는 것**입니다.

---

## 2. 머신러닝이 동작하는 과정
### 1) 데이터 수집
먼저 학습에 필요한 데이터를 모읍니다.

예:
- 집값 예측: 면적, 위치, 방 개수, 건축 연도, 실제 집값
- 스팸 분류: 이메일 내용, 스팸 여부

### 2) 데이터 전처리
그대로 쓰기 어려운 데이터는 정리합니다.

예:
- 결측치 처리
- 문자열을 숫자로 변환
- 이상치 제거
- 스케일 조정

### 3) 모델 선택
문제에 맞는 알고리즘을 고릅니다.

예:
- 선형회귀: 숫자 예측
- 로지스틱 회귀 / 의사결정트리: 분류
- 신경망: 복잡한 패턴 학습

### 4) 학습(Training)
모델이 데이터와 정답을 비교하면서 **오차를 줄이는 방향으로 내부 파라미터를 조정**합니다.

예:
- 예측값이 실제값보다 너무 크면 그 차이를 줄이도록 조정
- 이 과정을 수많은 데이터에 반복

### 5) 평가
학습한 모델이 **처음 보는 데이터에서도 잘 동작하는지** 확인합니다.

예:
- 정확도
- 정밀도
- 재현율
- 평균제곱오차(MSE)

### 6) 예

> **\*참고하십셔!** 위에서 쓴 `stream_graph`는 
>
> `LangGraph 공식 API가 아니라, 교재 저자 Teddy님이 만든 **`langchain-teddynote` 패키지의 출력용 헬퍼 함수**입니다. (`pyproject.toml`에 포함되어 `uv sync` 때 함께 설치됨)
>
> 내부적으로는 `agent.stream(inputs, stream_mode="messages")`로 토큰을 하나씩 받아, 노드 이름 구분선(`🔄 Node: model 🔄`)과 함께 보기 좋게 출력할 뿐입니다. 1주차 Models에서 배운 `stream()`을 에이전트에 적용한 것이라고 보면 됩니다.
>
> - **출력만 하고 반환값은 없습니다**(`None`). 결과를 변수에 담아 쓰려면 `agent.invoke(...)`를 사용하세요.

**위에서 잘 이해하셨지요? ModelRequest 만져봅시다!**

**ModelRequest 주요 속성:**

| 속성 | 설명 |
|:---|:---|
| `model` | 사용할 `BaseChatModel` 인스턴스 |
| `system_prompt` | 시스템 프롬프트 (선택적) |
| `messages` | 대화 메시지 리스트 (시스템 프롬프트 제외) |
| `tool_choice` | 도구 선택 설정 |
| `tools` | 사용 가능한 도구 리스트 |
| `response_format` | 응답 형식 지정 |
| `state` | 현재 에이전트 상태 (`AgentState`) |
| `runtime` | 에이전트 런타임 정보 |
| `model_settings` | 추가 모델 설정 (dict) |

아래 코드는 `override()` 메서드를 사용하여 여러 속성을 동시에 변경하는 예시입니다.

In [7]:
@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    """대화 복잡도에 따라 모델 선택"""
    message_count = len(request.state["messages"][-1].content)
    print(f"글자수: {message_count}")

    # 긴 대화에는 고급 모델 사용
    if message_count > 10:
        # 여러 속성 동시 변경
        new_request = request.override(
            model=advanced_model,
            system_prompt="emoji 를 사용해서 답변해줘",
            tool_choice="auto",
        )
    else:
        new_request = request.override(
            model=basic_model,
            system_prompt="한 문장으로 간결하게 답변해줘. emoji 는 사용하지 말아줘.",
            tool_choice="auto",
        )
    print(f"모델 선택: {new_request.model.model}")
    return handler(new_request)

    
agent = create_agent(
    model=basic_model, tools=[], middleware=[dynamic_model_selection]  # 기본 모델
)

### 글자수 기반 모델 선택 테스트

아래는 글자수 10자 미만일 때의 응답입니다. 간결한 답변을 생성하도록 설정되어 있습니다.

In [8]:
stream_graph(agent, inputs={"messages": [HumanMessage(content="머신러닝 동작원리")]})

글자수: 9
모델 선택: gpt-5.4-mini

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
머신러닝은 많은 데이터를 바탕으로 모델이 패턴을 학습해 입력과 출력의 관계를 스스로 추정하고, 새 데이터에 대해 예측이나 판단을 하는 방식으로 동작합니다.

아래는 글자수 10자 이상일 때의 응답입니다. 이모지를 사용하여 친근한 답변을 생성하도록 설정되어 있습니다.

In [9]:
stream_graph(
    agent,
    inputs={
        "messages": [
            HumanMessage(content="머신러닝의 동작 원리에 대해서 설명해 주세요.")
        ]
    },
)

글자수: 25
모델 선택: gpt-5.4

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
🤖 **머신러닝의 동작 원리**를 쉽게 설명해드릴게요.

---

## 1. 머신러닝이란? 📘
머신러닝은 **컴퓨터가 명시적으로 규칙을 일일이 프로그래밍하지 않아도**,  
**데이터를 통해 패턴을 학습**해서 예측이나 판단을 하도록 만드는 기술입니다.

예를 들어:

- 전통적인 프로그래밍 💻  
  - "이 조건이면 이렇게 처리해라"라는 규칙을 사람이 직접 작성
- 머신러닝 🧠  
  - 많은 데이터를 주면 컴퓨터가 그 안의 규칙을 스스로 찾아냄

---

## 2. 기본 동작 원리 🔧
머신러닝은 보통 다음 흐름으로 작동합니다.

### ① 데이터 수집 📂
먼저 학습에 필요한 데이터를 모읍니다.

예:
- 이메일 스팸 분류 → 스팸/정상 메일 데이터
- 집값 예측 → 집 크기, 위치, 방 개수, 실제 가격 데이터
- 이미지 인식 → 고양이/강아지 사진 데이터

👉 데이터는 머신러닝의 핵심입니다.  
좋은 데이터가 많을수록 성능이 좋아질 가능성이 큽니다.

---

### ② 특징(Feature) 입력 ✨
데이터 안에는 예측에 도움이 되는 정보가 있습니다. 이를 **특징(feature)**이라고 합니다.

예:
- 집값 예측:
  - 면적
  - 위치
  - 층수
  - 방 개수

머신러닝 모델은 이런 특징들을 바탕으로 결과를 예측합니다.

---

### ③ 모델 선택 🏗️
모델은 **입력과 출력 사이의 관계를 학습하는 수학적 함수**입니다.

예:
- 선형 회귀 📈
- 의사결정나무 🌳
- 신경망 🧠
- SVM
- 랜덤 포레스트 🌲

모델마다 잘하는 문제가 다릅니다.

---

### ④ 학습(Training) 🎓
이 단계에서 모델은 데이터의 패턴을 찾아갑니다.

예를 들어, 집값 예측 모델이라면:
- 입력: 집의 특징
- 정답: 실제 집값

모델은 처음에는 엉뚱하게 예측할 수 있지만 😅  
예측

---

## 프롬프트

에이전트의 동작을 제어하는 핵심 요소 중 하나는 시스템 프롬프트입니다. 시스템 프롬프트를 통해 에이전트의 역할, 응답 스타일, 제약 조건 등을 정의할 수 있습니다.

### 시스템 프롬프트

`system_prompt` 매개변수를 사용하여 에이전트의 기본 동작을 정의할 수 있습니다. 시스템 프롬프트는 모든 대화에서 일관되게 적용되며, 에이전트의 페르소나와 응답 가이드라인을 설정하는 데 사용됩니다.

아래 코드는 간결하고 정확한 응답을 생성하도록 시스템 프롬프트를 설정한 에이전트를 생성합니다.

In [10]:
# 다른 모델을 사용하려면 gpt-5.4-mini, claude-sonnet-5 등으로 변경하세요
model = init_chat_model("gpt-5.4")

agent = create_agent(
    model,
    system_prompt="You are a helpful assistant. Be concise and accurate.",
)

아래는 설정된 시스템 프롬프트를 사용한 에이전트의 응답 예시입니다.

In [10]:
stream_graph(
    agent,
    inputs={"messages": [HumanMessage(content="대한민국의 수도는 어디야?")]},
)

글자수: 14
모델 선택: gpt-5.4

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
대한민국의 수도는 **서울특별시**야! 🇰🇷🏙️

### 동적 시스템 프롬프트 (Dynamic Prompting)

런타임 컨텍스트나 에이전트 상태를 기반으로 시스템 프롬프트를 동적으로 생성해야 하는 경우가 있습니다. `dynamic_prompt` 데코레이터를 사용하면 요청마다 다른 시스템 프롬프트를 적용할 수 있습니다.

이 기능은 사용자 역할, 언어 설정, 응답 형식 등을 런타임에 결정해야 할 때 유용합니다. `context_schema`를 정의하여 에이전트 호출 시 필요한 컨텍스트 정보를 전달할 수 있습니다.

아래 코드는 답변 형식과 길이를 동적으로 설정하는 에이전트를 생성합니다.

In [11]:
from typing import TypedDict
from langchain.agents.middleware import dynamic_prompt, ModelRequest


class Context(TypedDict):
    prompt_type: str # "default" / "sns" / "article"
    length: int # 답변 글자 수 제한


@dynamic_prompt
def user_role_prompt(request: ModelRequest) -> str: #받는 인자: request 하나, 돌려줄 값: 시스템 프롬프트 문자열
    """사용자 역할에 따라 시스템 프롬프트 생성"""
    # 답변 형식 설정
    answer_type = (
        request.runtime.context.get("prompt_type", "default")
        if request.runtime.context
        else "default"
    )
    # 답변 길이 설정
    answer_length = (
        request.runtime.context.get("length", 20) 
        if request.runtime.context 
        else 20
    )
    base_prompt = "You are a helpful assistant. Answer in Korean.\n"

    # 답변 형식에 따라 시스템 프롬프트 생성(동적 프롬프팅)
    if answer_type == "default":
        return f"{base_prompt} [답변 형식] 간결하게 답변해줘. 답변 길이는 {answer_length}자 이하로 해줘."
    elif answer_type == "sns":
        return f"{base_prompt} [답변 형식] SNS 형식으로 답변해줘. 답변 길이는 {answer_length}자 이하로 해줘."
    elif answer_type == "article":
        return f"{base_prompt} [답변 형식] 뉴스 기사 형식으로 답변해줘. 답변 길이는 {answer_length}자 이하로 해줘."
    else:
        return f"{base_prompt} [답변 형식] 간결하게 답변해줘. 답변 길이는 {answer_length}자 이하로 해줘."


# 컨텍스트 스키마와 user_role_prompt 미들웨어를 사용하여 에이전트 생성
agent = create_agent(
    model=model,
    middleware=[user_role_prompt],
    context_schema=Context,
)

In [12]:
# 컨텍스트에 따라 시스템 프롬프트가 동적으로 설정됩니다
stream_graph(
    agent,
    inputs={
        "messages": [HumanMessage(content="머신러닝의 동작 원리에 대해서 설명해줘")]
    },
    context=Context(prompt_type="article", length=1000),
)


🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
[과학·기술] 머신러닝, “데이터로 규칙을 찾는 기술”… 작동 원리는?

머신러닝은 컴퓨터가 명확한 규칙을 일일이 입력받지 않아도, 많은 데이터 속에서 패턴을 찾아 스스로 예측하거나 판단하도록 만드는 기술이다. 전통적인 프로그래밍이 “규칙+데이터=결과” 방식이라면, 머신러닝은 “데이터+결과=규칙 학습”에 가깝다.

작동 원리의 핵심은 학습이다. 먼저 컴퓨터는 입력 데이터와 정답이 포함된 학습용 자료를 받는다. 이후 알고리즘은 예측값과 실제 정답의 차이, 즉 오차를 계산한다. 이 오차를 줄이는 방향으로 내부 파라미터를 조금씩 조정하면서 더 나은 모델을 만든다. 이 과정이 반복되면 컴퓨터는 데이터 속 경향성을 점점 더 잘 이해하게 된다.

머신러닝은 크게 지도학습, 비지도학습, 강화학습으로 나뉜다. 지도학습은 정답이 있는 데이터를 통해 분류나 예측을 수행하고, 비지도학습은 정답 없이 데이터의 구조나 군집을 찾는다. 강화학습은 보상과 벌점을 바탕으로 최적의 행동 전략을 익히는 방식이다.

전문가들은 머신러닝의 성능이 데이터의 양과 질, 그리고 모델 설계에 크게 좌우된다고 설명한다. 학습이 끝난 모델은 새로운 데이터에도 적용되지만, 학습 데이터에만 지나치게 맞춰진 ‘과적합’은 주의해야 할 문제로 꼽힌다.

결국 머신러닝은 데이터를 통해 경험을 축적하고, 그 경험을 바탕으로 미래를 예측하는 기술로 평가된다.

In [13]:
stream_graph(
    agent,
    inputs={
        "messages": [HumanMessage(content="머신러닝의 동작 원리에 대해서 설명해줘")]
    },
    context=Context(prompt_type="sns", length=50),
)


🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
🤖 머신러닝: 데이터를 보고 패턴 학습→예측·판단!

---

## 미들웨어

미들웨어를 사용하면 모델 호출 전후에 커스텀 로직을 실행할 수 있습니다. `@before_model` 및 `@after_model` 데코레이터를 사용하여 모델 호출을 감싸는 훅을 정의할 수 있습니다.

**미들웨어 활용 사례:**
- 모델 호출 전 메시지 전처리 (예: 쿼리 재작성)
- 모델 호출 후 응답 후처리 (예: 필터링, 로깅)
- 상태 기반 동적 라우팅

아래 코드는 모델 호출 전후에 로깅을 수행하는 미들웨어 예시입니다.

In [14]:
from langchain.agents.middleware import (
    before_model,
    after_model,
)
from langchain.agents.middleware import (
    AgentState,
    ModelRequest,
    ModelResponse,
)
from langchain_core.prompts import PromptTemplate
from langgraph.runtime import Runtime
from typing import Any, Callable


# 노드 스타일: 모델 호출 전 로깅
@before_model
def log_before_model(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    print(
        f"\033[95m\n\n모델 호출 전 메시지 {len(state['messages'])}개가 있습니다\033[0m"
    )
    last_message = state["messages"][-1].content
    # 다른 모델을 사용하려면 gpt-5.4-mini, claude-sonnet-5 등으로 변경하세요
    llm = init_chat_model("gpt-5.4")

    query_rewrite = (
        PromptTemplate.from_template(
            "Rewrite the following query to be more understandable. Do not change the original meaning. Make it one sentence: {query}"
        )
        | llm
    )
    rewritten_query = query_rewrite.invoke({"query": last_message})

    return {"messages": [rewritten_query.content]}


@after_model
def log_after_model(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:

    print(
        f"\033[95m\n\n모델 호출 후 메시지 {len(state['messages'])}개가 있습니다\033[0m"
    )
    for i, message in enumerate(state["messages"]):
        print(f"[{i}] {message.content}")
    return None

> **\*참고하십셔!** 왜 `@before_model` / `@after_model`을 "노드 스타일"이라고 부를까?
>
> 1주차 QuickStart에서 직접 만든 챗봇 노드를 다시 보겠습니다.
>
> ```python
> class State(TypedDict):
>     messages: Annotated[list, add_messages]
>
> def chatbot(state: State):
>     response = llm.invoke(state["messages"])
>     return {"messages": [response]}      # 상태 업데이트 dict 반환
>
> graph_builder.add_node("chatbot", chatbot)
> ```
>
> 위의 `log_before_model`과 모양이 같습니다. **`state`를 받고, 바꿀 부분만 `dict`로 돌려주면** 리듀서가 기존 상태에 합칩니다. 실제로 `create_agent`는 이 함수들을 그래프에 **별도 노드로 끼워 넣습니다.**
>
> ```
> START → [log_before_model] → [model] → [log_after_model] → END
> ```
>
> | | 1·2주차 노드 함수 | `@before_model` / `@after_model` |
> |:---|:---|:---|
> | 받는 인자 | `state` (+ 필요하면 `runtime`) | `state`, `runtime` |
> | 반환 | 상태 업데이트 `dict` | 상태 업데이트 `dict` 또는 `None`(변경 없음) |
> | 그래프 연결 | `add_node`, `add_edge`로 직접 | `middleware=[...]`에 넣으면 자동 |
>
> **그럼 `messages`의 `add_messages` 리듀서는 어디서 선언됐을까?** 위 코드에는 `State`를 정의한 곳이 없습니다. `create_agent`가 내부적으로 쓰는 기본 상태 `AgentState`에 이미 선언되어 있습니다. (`langchain/agents/middleware/types.py`)
>
> ```python
> class AgentState(TypedDict):
>     messages: Required[Annotated[list[AnyMessage], add_messages]]
>     ...
> ```
>
> 그래서 `log_before_model`이 `{"messages": [다듬은 질문]}`을 반환하면 원래 질문을 **덮어쓰지 않고 뒤에 추가**합니다. (id가 같은 메시지를 반환해야 교체 — 1주차 `add_messages` 참고)

In [15]:
agent = create_agent(
    model,
    middleware=[
        log_before_model,
        log_after_model,
    ],
)

In [16]:
stream_graph(
    agent,
    inputs={"messages": [HumanMessage(content="대한민국 수도")]},
)



모델 호출 전 메시지 1개가 있습니다

🔄 Node: log_before_model.before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
대한민국의 수도는 어디인가요?
🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
대한민국의 수도는 **서울특별시**입니다.

모델 호출 후 메시지 3개가 있습니다
[0] 대한민국 수도
[1] 대한민국의 수도는 어디인가요?
[2] 대한민국의 수도는 **서울특별시**입니다.


### 클래스 기반 미들웨어

데코레이터 대신 클래스 기반 미들웨어를 사용할 수 있습니다. `AgentMiddleware` 클래스를 상속하고 `before_model` 및 `after_model` 메서드를 오버라이드하여 커스텀 로직을 구현합니다.

클래스 기반 미들웨어는 커스텀 상태 스키마를 정의하거나 복잡한 미들웨어 로직을 구조화할 때 유용합니다.

아래 코드는 클래스 기반 미들웨어를 사용하여 커스텀 상태를 관리하는 예시입니다.

In [18]:
from typing import Any
from langchain.agents import AgentState
from langchain.agents.middleware import AgentMiddleware


# 커스텀 상태 스키마 정의
class CustomState(AgentState):
    user_preferences: dict


class CustomMiddleware(AgentMiddleware):
    state_schema = CustomState
    tools = []

    def before_model(self, state: CustomState, runtime) -> dict[str, Any] | None:
        # 모델 호출 전 커스텀 로직
        pass


agent = create_agent(model, tools=[], middleware=[CustomMiddleware()])

# 에이전트는 이제 메시지 외에 추가 상태를 추적할 수 있습니다
result = agent.invoke(
    {
        "messages": [{"role": "user", "content": "I prefer technical explanations"}],
        "user_preferences": {"style": "technical", "verbosity": "detailed"},
    }
)

### 모델 오류 시 재시도 로직

`wrap_model_call` 데코레이터를 사용하면 모델 호출 실패 시 자동으로 재시도하는 로직을 구현할 수 있습니다. 이는 네트워크 오류나 일시적인 API 장애에 대응하는 데 유용합니다.

아래 코드는 최대 3회까지 재시도하는 미들웨어 예시입니다.

In [17]:
@wrap_model_call
def retry_model(request: ModelRequest, handler) -> ModelResponse:
    for attempt in range(3):
        try:
            return handler(request)
        except Exception as e:
            if attempt == 2:
                raise
            print(f"오류 발생으로 {attempt + 1}/3 번째 재시도합니다: {e}")

In [18]:
# 다른 모델을 사용하려면 gpt-5.4-mini, claude-sonnet-5 등으로 변경하세요
# 일부러 존재하지 않는 모델명을 사용하여 재시도 로직을 테스트합니다
model = init_chat_model("gpt-5.4-invalid")

agent = create_agent(
    model,
    middleware=[retry_model],
)

In [19]:
stream_graph(
    agent,
    inputs={"messages": [HumanMessage(content="대한민국의 수도는?")]},
)

오류 발생으로 1/3 번째 재시도합니다: Error code: 404 - {'error': {'message': 'The model `gpt-5.4-invalid` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}
오류 발생으로 2/3 번째 재시도합니다: Error code: 404 - {'error': {'message': 'The model `gpt-5.4-invalid` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}


OpenAIModelNotFoundError: Error code: 404 - {'error': {'message': 'The model `gpt-5.4-invalid` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}

---

# 🧩 3주차 퀴즈 ① — Agents & Middleware

이 노트북에서 배운 내용을 직접 실습해보는 퀴즈입니다. 빈칸(`________`)을 채우고 셀을 실행해서 **"통과! 🎉"** 가 출력되면 성공입니다.

**제출 방법**

1. 퀴즈는 `submissions/본인이름/` 폴더에 미리 들어 있는 **본인 이름이 붙은 노트북**에서 풉니다. (`notebooks/` 원본은 수정 금지!)
2. 각 퀴즈의 빈칸을 채우고 실행 결과를 확인합니다.
3. `week3/본인이름` 브랜치에서 커밋 후 Pull Request를 올립니다. (자세한 방법: 3주차 README)

> ✅ 이번 퀴즈는 **API 키 없이** 전부 풀 수 있습니다. 실제 LLM 대신 아래의 **가짜 모델(`EchoModel`)** 을 사용합니다.

### 준비: 가짜 모델 `EchoModel`

아래 셀은 **수정하지 말고 실행만** 하세요.

`EchoModel`은 LLM을 부르는 대신, **자기가 받은 요청을 그대로 보고하는** 가짜 모델입니다. 응답 형식은 다음과 같습니다.

```
[모델이름] 받은 메시지 N개 | 시스템: (받은 시스템 프롬프트)
```

그래서 응답만 보면 **어떤 모델이 불렸는지, 어떤 시스템 프롬프트가 들어갔는지, 메시지가 몇 개였는지** 확인할 수 있습니다. 미들웨어가 요청을 제대로 고쳤는지 채점할 때 이 정보를 사용합니다.

In [20]:
from typing import Any, TypedDict

from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.outputs import ChatGeneration, ChatResult


class EchoModel(BaseChatModel):
    # 받은 요청을 그대로 보고하는 가짜 모델 (API 키 불필요)
    tag: str = "echo"

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        system = next((m.content for m in messages if m.type == "system"), "")
        n = sum(1 for m in messages if m.type != "system")
        reply = f"[{self.tag}] 받은 메시지 {n}개 | 시스템: {system}"
        return ChatResult(generations=[ChatGeneration(message=AIMessage(content=reply))])

    @property
    def _llm_type(self) -> str:
        return "echo"


basic_model = EchoModel(tag="basic")
advanced_model = EchoModel(tag="advanced")

print(basic_model.invoke("안녕").content)

[basic] 받은 메시지 1개 | 시스템: 


### Q1. 미들웨어 함수는 누가 호출할까?

`dynamic_model_selection(request, handler)`를 정의만 했는데, `request`와 `handler`에는 누가 값을 넣어줄까요? 정답은 **에이전트**입니다. 에이전트가 LLM을 부르기 직전에 우리 함수를 **대신 호출**하면서 두 값을 채워 넣습니다.

에이전트 없이 이 구조를 **순수 Python으로 흉내** 내봅시다. 빈칸을 채우세요.
(힌트: `handler`에는 값이 아니라 **함수**가 들어옵니다)

In [23]:
# ① 우리가 작성하는 미들웨어: 요청을 고친 뒤, 넘겨받은 함수로 '진짜 호출'을 맡긴다
def my_middleware(request, handler):
    request = request + " (수정됨)"
    return handler(request)


# ② 실제 모델 호출 역할 (에이전트 내부에 있는 것)
def call_model(request):
    return f"응답<{request}>"


# ③ 에이전트가 내부에서 하는 일: 미들웨어를 '대신' 호출하며 request와 handler를 채워 넣는다
def agent_invoke(user_input, middleware):
    return middleware(user_input, call_model)


result = agent_invoke("안녕", my_middleware)
print(result)

# ---- 자동 채점 ----
assert result == "응답<안녕 (수정됨)>", "미들웨어가 요청을 고친 뒤, handler로 넘겨 호출해야 합니다"
print("통과! 🎉")

응답<안녕 (수정됨)>
통과! 🎉


### Q2. `wrap_model_call`로 모델 갈아끼우기

마지막 질문이 **10자 초과**면 `advanced_model`, 아니면 `basic_model`을 쓰는 미들웨어를 완성하세요.

- 모델은 `request.model = ...`로 직접 바꾸지 말고, **`override()`** 로 새 요청을 만드세요. (직접 대입은 현재 버전에서 deprecated 경고가 뜹니다)
- 마지막에 무엇을 `return` 해야 실제로 모델이 호출될까요?

In [26]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse


@wrap_model_call
def pick_model(request: ModelRequest, handler) -> ModelResponse:
    last_question = request.state["messages"][-1].content

    if len(last_question) > 10:
        new_request = request.override(model=advanced_model)
    else:
        new_request = request.override(model=basic_model)

    return handler(new_request)


agent = create_agent(basic_model, tools=[], middleware=[pick_model])

short = agent.invoke({"messages": [HumanMessage("안녕")]})["messages"][-1].content
long_ = agent.invoke({"messages": [HumanMessage("머신러닝의 동작 원리를 설명해 주세요")]})["messages"][-1].content
print(short)
print(long_)

# ---- 자동 채점 ----
assert short.startswith("[basic]"), "짧은 질문은 basic_model이 받아야 합니다"
assert long_.startswith("[advanced]"), "긴 질문은 advanced_model이 받아야 합니다"
print("통과! 🎉")

[basic] 받은 메시지 1개 | 시스템: 
[advanced] 받은 메시지 1개 | 시스템: 
통과! 🎉


### Q3. `@dynamic_prompt` + `context`

호출할 때 넘긴 `context`의 `tone` 값으로 시스템 프롬프트를 만드는 미들웨어를 완성하세요.

- `@dynamic_prompt` 함수는 `handler` 없이 **`request` 하나**만 받고, **시스템 프롬프트 문자열**을 돌려줍니다.
- 호출할 때 넘길 context의 형식(`ToneContext`)을 에이전트에 **등록**해야 합니다. (본문의 동적 시스템 프롬프트 예제 참고)
- context를 **안 넘기면** `"기본"` 말투가 되어야 합니다. (context는 우리가 넣어줘야만 들어옵니다!)

In [27]:
from langchain.agents.middleware import dynamic_prompt


class ToneContext(TypedDict):
    tone: str


@dynamic_prompt
def tone_prompt(request: ModelRequest) -> str:
    # 호출할 때 넘긴 context는 request.runtime.context에 들어 있습니다. 3주차 두번째 노트북에서 자세히 나옵니당!
    ctx = request.runtime.context
    tone = ctx.get("tone", "기본") if ctx else "기본"
    return f"{tone} 말투로 답변해줘."


agent = create_agent(
    basic_model,
    tools=[],
    middleware=[tone_prompt],
    context_schema=ToneContext,
)

# 1) context를 넘겨서 호출
with_context = agent.invoke(
    {"messages": [HumanMessage("안녕")]},
    context=ToneContext(tone="해적"),
)["messages"][-1].content

# 2) context 없이 호출
without_context = agent.invoke({"messages": [HumanMessage("안녕")]})["messages"][-1].content

print(with_context)
print(without_context)

# ---- 자동 채점 ----
assert "시스템: 해적 말투로 답변해줘." in with_context, "context의 tone이 시스템 프롬프트에 들어가야 합니다"
assert "시스템: 기본 말투로 답변해줘." in without_context, "context가 없으면 '기본' 말투여야 합니다"
print("통과! 🎉")

[basic] 받은 메시지 1개 | 시스템: 해적 말투로 답변해줘.
[basic] 받은 메시지 1개 | 시스템: 기본 말투로 답변해줘.
통과! 🎉


### Q4. `@before_model`: 메시지 "추가" vs "교체"

`@before_model`은 **노드 스타일** 미들웨어입니다. 1·2주차 노드 함수처럼 `state`를 받고, 바꿀 부분만 `dict`로 돌려줍니다.

그리고 `messages`에는 `AgentState`에 선언된 **`add_messages` 리듀서**가 걸려 있습니다. 그래서 무엇을 돌려주느냐에 따라 결과가 달라집니다.

- **(a) 추가**: 새 메시지를 그냥 돌려주면 → 원래 질문 **뒤에 붙습니다**
- **(b) 교체**: 원래 메시지와 **같은 `id`** 로 돌려주면 → 원래 질문을 **덮어씁니다** (1주차 퀴즈 ① Q3 복습)


In [31]:
from langchain.agents.middleware import before_model, AgentState
from langgraph.runtime import Runtime


# (a) 추가: 질문 뒤에 "존댓말로 답해주세요"가 붙은 새 질문을 추가
@before_model
def add_polite(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    last = state["messages"][-1]
    return {"messages": [HumanMessage(f"{last.content} (존댓말로 답해주세요)")]}


# (b) 교체: 같은 id를 써서 원래 질문을 덮어쓰기
@before_model
def replace_polite(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    last = state["messages"][-1]
    return {"messages": [HumanMessage(f"{last.content} (존댓말로 답해주세요)", id=last.id)]}


out_a = create_agent(basic_model, tools=[], middleware=[add_polite]).invoke(
    {"messages": [HumanMessage("대한민국 수도")]}
)
out_b = create_agent(basic_model, tools=[], middleware=[replace_polite]).invoke(
    {"messages": [HumanMessage("대한민국 수도")]}
)

print("(a)", [m.content for m in out_a["messages"]])
print("(b)", [m.content for m in out_b["messages"]])

# ---- 자동 채점 ----
assert len(out_a["messages"]) == 3, "(a) 원래 질문 + 추가된 질문 + AI 응답 = 3개여야 합니다"
assert "받은 메시지 2개" in out_a["messages"][-1].content, "(a) 모델은 질문 2개를 받아야 합니다"
assert len(out_b["messages"]) == 2, "(b) 교체된 질문 + AI 응답 = 2개여야 합니다"
assert out_b["messages"][0].content == "대한민국 수도 (존댓말로 답해주세요)", "(b) 원래 질문이 교체되어야 합니다"
print("통과! 🎉")

(a) ['대한민국 수도', '대한민국 수도 (존댓말로 답해주세요)', '[basic] 받은 메시지 2개 | 시스템: ']
(b) ['대한민국 수도 (존댓말로 답해주세요)', '[basic] 받은 메시지 1개 | 시스템: ']
통과! 🎉


### Q5. 개념 확인 (주관식)

이 셀을 **더블클릭**해서 아래 질문에 직접 답을 작성하세요.

1. Q2의 `pick_model`에서 마지막 줄을 `return handler(new_request)` 대신 **`handler(new_request)`** (return 없이)로 쓰면 아래 오류가 납니다. 왜 그럴까요? 그리고 `handler(...)`를 **아예 호출하지 않으면** 어떻게 될까요?
   ```
   AttributeError: 'NoneType' object has no attribute 'result'
   ```
   - 답: middleware에서 pick_model의 반환값을 사용하려는데, None값이 나와서 오류가 납니다. handler 를 호출하지 않으면 미들웨어가 작동을 안해서 오류가 날 것 같습니다...

2. `@wrap_model_call`, `@dynamic_prompt`, `@before_model`은 **받는 인자와 돌려줄 값**이 서로 다릅니다. Q2~Q4에서 쓴 세 함수를 떠올리며 표를 채워보세요.

   | 데코레이터 | 받는 인자 | 돌려줄 값 |
   |:---|:---|:---|
   | `@wrap_model_call` | request, handler | handler(request) |
   | `@dynamic_prompt` | request | agent.invoke의 값 |
   | `@before_model` | state, runtime | dict |

3. `state`와 `context`는 둘 다 에이전트에 값을 전달하지만 역할이 다릅니다. **누가 채우는지**, **실행 중에 바뀌는지** 기준으로 비교하세요! 
   - 답: state는 실행자가 채우고 그에 따라서 실행 중에 업데이트가 됩니다. 반면 context는 설계자?가 채워놓은 것이고, 고정값으로 사용하기에 실행 중 바뀌지 않습니다. 즉, 전자는 작업 상태 등을 표현할 때 사용할 수 있고, 후자는 작업에 도움?이 되도록 정보를 표현할 때 사용할 수 있습니다.